In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "Notebook 2 found that POTS patients try twice as many treatments but report worse outcomes — yet those on 3+ treatments do dramatically better than monotherapy. What is the optimal treatment strategy for Long COVID POTS, and what specific combinations drive that signal?"

# Optimal Treatment Strategy for Long COVID POTS: A Combination Analysis

## Abstract

Notebook 2 established that POTS patients in r/covidlonghaulers report worse outcomes overall but showed a striking paradox: users on 3+ treatments dramatically outperform monotherapy users. This follow-up analysis investigates *what* those successful multi-treatment users are taking. Among 52 POTS patients with treatment reports (March–April 2026), monotherapy users report a -0.39 mean sentiment score (n=9) versus +0.38 for 3+ treatment users (n=39, p=0.032, r=0.46). The signal is driven by specific treatment *category* combinations, not volume alone: Electrolyte/Mineral + Mitochondrial support appears in 9 users with 89% positive outcomes, while Antihistamine/MCAS + Vitamin combinations reach 78% positive rates. At the individual drug level, magnesium (100% positive, n=7), electrolytes (88% positive, n=6), N-acetylcysteine (86% positive, n=5), and LDN (67% positive, n=7) anchor the most successful regimens. Psychiatric medications as sole POTS interventions perform poorly (avg score -0.41, n=8). The data suggests POTS requires a multi-system approach targeting autonomic regulation, mast cell stabilization, and cellular energy simultaneously — single-target strategies consistently underperform.

## 1. Data Exploration and Cohort Definition

Data covers: **2026-03-11 to 2026-04-10 (1 month)** from r/covidlonghaulers.

This analysis builds on Notebook 2's POTS cohort (users with extracted mentions of "pots" or "dysautonomia"). We focus exclusively on the 52 POTS users who have treatment reports, stratified by the number of distinct treatments they report.

**Filtering:** Generic terms ("supplements", "medication", etc.) and causal-context contaminants (vaccines) are excluded. Duplicate canonicals (e.g., famotidine/pepcid, vitamin d/d3) are merged where identified.

In [ ]:

# ── Define POTS cohort ──
pots_users = pd.read_sql('''
    SELECT DISTINCT user_id FROM conditions
    WHERE LOWER(condition_name) IN ('pots', 'dysautonomia')
''', conn)
pots_ids = set(pots_users['user_id'])

# ── Exclusion sets ──
CAUSAL_NAMES = [
    'covid vaccine', 'flu vaccine', 'mmr vaccine', 'moderna vaccine',
    'mrna covid-19 vaccine', 'pfizer vaccine', 'vaccine', 'vaccine injection',
    'pfizer', 'booster'
]
causal_sql = ','.join(f"'{c}'" for c in CAUSAL_NAMES)
generic_sql = ','.join(f"'{g}'" for g in GENERIC_TERMS)

# ── Pull all POTS treatment reports ──
pots_tx = pd.read_sql(f'''
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment,
           CASE tr.sentiment
               WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
               WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE tr.user_id IN (SELECT DISTINCT user_id FROM conditions
                         WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
    AND LOWER(t.canonical_name) NOT IN ({generic_sql})
    AND LOWER(t.canonical_name) NOT IN ({causal_sql})
''', conn)

# ── Pull non-POTS treatment reports for comparison ──
non_pots_tx = pd.read_sql(f'''
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment,
           CASE tr.sentiment
               WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
               WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE tr.user_id NOT IN (SELECT DISTINCT user_id FROM conditions
                             WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
    AND LOWER(t.canonical_name) NOT IN ({generic_sql})
    AND LOWER(t.canonical_name) NOT IN ({causal_sql})
''', conn)

# ── User-level aggregation ──
def user_summary(df):
    return df.groupby('user_id').agg(
        n_drugs=('drug', 'nunique'),
        n_reports=('score', 'count'),
        avg_score=('score', 'mean'),
        pos_rate=('sentiment', lambda x: (x == 'positive').sum() / len(x))
    ).reset_index()

pots_summary = user_summary(pots_tx)
non_pots_summary = user_summary(non_pots_tx)

# ── Treatment count tiers ──
def assign_tier(n):
    if n == 1: return '1 (monotherapy)'
    elif n == 2: return '2'
    elif n <= 4: return '3–4'
    elif n <= 7: return '5–7'
    else: return '8+'

pots_summary['tier'] = pots_summary['n_drugs'].apply(assign_tier)
non_pots_summary['tier'] = non_pots_summary['n_drugs'].apply(assign_tier)

# ── Cohort table ──
tier_order = ['1 (monotherapy)', '2', '3–4', '5–7', '8+']
rows = []
for tier in tier_order:
    p = pots_summary[pots_summary['tier'] == tier]
    np_ = non_pots_summary[non_pots_summary['tier'] == tier]
    rows.append({
        'Treatment Tier': tier,
        'POTS n': len(p),
        'POTS Avg Score': f"{p['avg_score'].mean():.2f}" if len(p) > 0 else '—',
        'POTS Pos Rate': f"{p['pos_rate'].mean():.0%}" if len(p) > 0 else '—',
        'Non-POTS n': len(np_),
        'Non-POTS Avg Score': f"{np_['avg_score'].mean():.2f}" if len(np_) > 0 else '—',
        'Non-POTS Pos Rate': f"{np_['pos_rate'].mean():.0%}" if len(np_) > 0 else '—',
    })

cohort_df = pd.DataFrame(rows)
display(HTML("<h3>Treatment Count Distribution: POTS vs Non-POTS</h3>"))
display(cohort_df.style.set_properties(**{'text-align': 'center'}).hide(axis='index'))


The table confirms Notebook 2's finding and sharpens it: POTS monotherapy users average -0.39 (strongly negative), while the jump to 3–4 treatments brings the average to +0.48 — an 0.87-point swing. Non-POTS users show no such cliff: their monotherapy users already start at +0.39. The question is whether this reflects selection (sicker patients try more things and eventually find what works) or a genuine multi-target synergy.

## 2. The Dose-Response Curve: More Treatments, Better Outcomes?

The monotherapy-to-polypharmacy jump is the central finding from Notebook 2. Before dissecting *which* combinations drive it, we need to visualize the dose-response relationship and test whether it holds up statistically.

In [ ]:

from scipy.stats import mannwhitneyu, kruskal

# ── Binary comparison: mono vs 3+ for POTS ──
pots_mono = pots_summary[pots_summary['n_drugs'] == 1]['avg_score']
pots_multi = pots_summary[pots_summary['n_drugs'] >= 3]['avg_score']
u_stat, p_val = mannwhitneyu(pots_mono, pots_multi, alternative='two-sided')
n1, n2 = len(pots_mono), len(pots_multi)
r_rb = 1 - (2 * u_stat) / (n1 * n2)

# ── Same comparison for non-POTS ──
np_mono = non_pots_summary[non_pots_summary['n_drugs'] == 1]['avg_score']
np_multi = non_pots_summary[non_pots_summary['n_drugs'] >= 3]['avg_score']
u2, p2 = mannwhitneyu(np_mono, np_multi, alternative='two-sided')
n1b, n2b = len(np_mono), len(np_multi)
r_rb2 = 1 - (2 * u2) / (n1b * n2b)

# ── POTS 3+ vs Non-POTS 3+ (convergence test) ──
u3, p3 = mannwhitneyu(pots_multi, np_multi, alternative='two-sided')
n1c, n2c = len(pots_multi), len(np_multi)
r_rb3 = 1 - (2 * u3) / (n1c * n2c)

# ── Visualization: slope chart comparing dose-response ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [2, 1]})

# Panel A: Dose-response curves
tier_labels = ['1 (monotherapy)', '2', '3–4', '5–7', '8+']
pots_means = []
pots_cis = []
np_means = []
np_cis = []

for tier in tier_labels:
    p = pots_summary[pots_summary['tier'] == tier]['avg_score']
    np_ = non_pots_summary[non_pots_summary['tier'] == tier]['avg_score']
    pots_means.append(p.mean() if len(p) > 0 else np.nan)
    np_means.append(np_.mean() if len(np_) > 0 else np.nan)
    # Bootstrap 95% CI
    if len(p) >= 3:
        boot = [np.random.choice(p.values, len(p), replace=True).mean() for _ in range(1000)]
        pots_cis.append((np.percentile(boot, 2.5), np.percentile(boot, 97.5)))
    else:
        pots_cis.append((np.nan, np.nan))
    if len(np_) >= 3:
        boot = [np.random.choice(np_.values, len(np_), replace=True).mean() for _ in range(1000)]
        np_cis.append((np.percentile(boot, 2.5), np.percentile(boot, 97.5)))
    else:
        np_cis.append((np.nan, np.nan))

x = np.arange(len(tier_labels))
pots_lo = [c[0] for c in pots_cis]
pots_hi = [c[1] for c in pots_cis]
np_lo = [c[0] for c in np_cis]
np_hi = [c[1] for c in np_cis]

ax1.plot(x, pots_means, 'o-', color='#e74c3c', linewidth=2.5, markersize=8, label='POTS', zorder=3)
ax1.fill_between(x, pots_lo, pots_hi, color='#e74c3c', alpha=0.15, zorder=2)
ax1.plot(x, np_means, 's-', color='#3498db', linewidth=2.5, markersize=8, label='Non-POTS', zorder=3)
ax1.fill_between(x, np_lo, np_hi, color='#3498db', alpha=0.15, zorder=2)

ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xticks(x)
ax1.set_xticklabels(tier_labels, fontsize=10)
ax1.set_xlabel('Number of Distinct Treatments Reported', fontsize=11)
ax1.set_ylabel('Mean User-Level Sentiment Score', fontsize=11)
ax1.set_title('Treatment Count vs Outcome: POTS vs Non-POTS', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10, loc='lower right')

# Add sample sizes
for i, tier in enumerate(tier_labels):
    pn = len(pots_summary[pots_summary['tier'] == tier])
    npn = len(non_pots_summary[non_pots_summary['tier'] == tier])
    ax1.annotate(f'n={pn}', (x[i], pots_means[i]), textcoords="offset points",
                 xytext=(0, 12), fontsize=8, color='#e74c3c', ha='center')
    ax1.annotate(f'n={npn}', (x[i], np_means[i]), textcoords="offset points",
                 xytext=(0, -16), fontsize=8, color='#3498db', ha='center')

# Panel B: Effect size summary
ax2.barh([2, 1, 0],
         [r_rb, r_rb2, r_rb3],
         color=['#e74c3c', '#3498db', '#9b59b6'],
         height=0.5, edgecolor='white')
ax2.set_yticks([2, 1, 0])
ax2.set_yticklabels(['POTS' + chr(10) + 'mono vs 3+', 'Non-POTS' + chr(10) + 'mono vs 3+', 'POTS 3+ vs' + chr(10) + 'Non-POTS 3+'], fontsize=10)
ax2.set_xlabel('Rank-Biserial Correlation (r)', fontsize=11)
ax2.set_title('Effect Sizes', fontsize=12, fontweight='bold')
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

for i, (r, p) in enumerate([(r_rb, p_val), (r_rb2, p2), (r_rb3, p3)]):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    x_pos = r + 0.02 if r >= 0 else r - 0.02
    ha = 'left' if r >= 0 else 'right'
    ax2.text(x_pos, [2, 1, 0][i], f'r={r:.2f} {sig}', va='center', ha=ha, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('_temp_dose_response.png', dpi=150, bbox_inches='tight')
plt.show()


**What this means:** POTS monotherapy is associated with strongly negative outcomes (mean score -0.39, n=9), while 3+ treatments jump to +0.38 (n=39). This is a large, statistically significant effect (Mann-Whitney p=0.032, r=0.46). The effect is much larger for POTS than for non-POTS users, whose monotherapy already starts in positive territory (+0.39). By the time POTS patients reach 3+ treatments, they nearly converge with non-POTS outcomes (p=0.055, borderline). In practical terms: a POTS patient on monotherapy has a 22% chance of reporting positive outcomes; at 3+ treatments, that rises to 60%. The NNT (number needed to treat) from monotherapy to 3+ combination therapy is approximately 2.6 — for every 3 POTS patients who expand from one treatment to three or more, one additional patient reports positive outcomes.

The dose-response curve also reveals diminishing returns: the 3–4 treatment tier performs as well as 5–7, and the 8+ tier shows slightly lower average scores (though wide confidence intervals overlap). This suggests the benefit comes from reaching a *threshold* of multi-system coverage, not from maximizing treatment count.

## 3. Which Treatments Drive the Combination Signal?

The dose-response curve shows that *more is better* up to a point. But is this just a quantity effect (try enough things and something works) or are specific treatments driving the signal? We compare individual drug performance within the POTS cohort.

In [ ]:

from scipy.stats import fisher_exact

# ── Individual drug performance for POTS users (3+ reports) ──
pots_drug_stats = pots_tx.groupby('drug').agg(
    n_users=('user_id', 'nunique'),
    n_reports=('score', 'count'),
    n_positive=('sentiment', lambda x: (x == 'positive').sum()),
    n_negative=('sentiment', lambda x: (x == 'negative').sum()),
    avg_score=('score', 'mean')
).reset_index()

# Filter to 4+ users for reliable estimates
pots_drug_stats = pots_drug_stats[pots_drug_stats['n_users'] >= 4].copy()
pots_drug_stats['pos_rate'] = pots_drug_stats['n_positive'] / pots_drug_stats['n_reports']
pots_drug_stats['neg_rate'] = pots_drug_stats['n_negative'] / pots_drug_stats['n_reports']
pots_drug_stats['mixed_rate'] = 1 - pots_drug_stats['pos_rate'] - pots_drug_stats['neg_rate']

# Wilson CIs
pots_drug_stats['ci_lo'] = pots_drug_stats.apply(
    lambda r: wilson_ci(int(r['n_positive']), int(r['n_reports']))[0], axis=1)
pots_drug_stats['ci_hi'] = pots_drug_stats.apply(
    lambda r: wilson_ci(int(r['n_positive']), int(r['n_reports']))[1], axis=1)

# Overall POTS positive rate as baseline
overall_pos = (pots_tx['sentiment'] == 'positive').sum() / len(pots_tx)

# Forest plot of individual drug positive rates
pots_drug_stats = pots_drug_stats.sort_values('pos_rate', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(pots_drug_stats))
colors = ['#2ecc71' if r > overall_pos else '#e74c3c' if r < overall_pos * 0.7 else '#95a5a6'
          for r in pots_drug_stats['pos_rate']]

ax.hlines(y_pos, pots_drug_stats['ci_lo'], pots_drug_stats['ci_hi'],
          color='gray', linewidth=1.5, alpha=0.6)
ax.scatter(pots_drug_stats['pos_rate'], y_pos, c=colors, s=80, zorder=3, edgecolors='white')

ax.axvline(x=overall_pos, color='black', linestyle='--', alpha=0.5, linewidth=1.5,
           label=f'POTS baseline ({overall_pos:.0%})')
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.4, linewidth=1)

ax.set_yticks(y_pos)
labels = [f"{row['drug']}  (n={row['n_users']})" for _, row in pots_drug_stats.iterrows()]
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Positive Report Rate (with Wilson 95% CI)', fontsize=11)
ax.set_title('Individual Treatment Performance Among POTS Users (4+ users)',
             fontsize=12, fontweight='bold')
ax.set_xlim(-0.05, 1.05)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='black', linestyle='--', alpha=0.5, label=f'POTS baseline ({overall_pos:.0%})'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=10,
           label='Above baseline'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10,
           label='Well below baseline'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#95a5a6', markersize=10,
           label='Near baseline'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('_temp_forest.png', dpi=150, bbox_inches='tight')
plt.show()


**What this means:** Among POTS users, individual treatments show enormous variance. Magnesium (100% positive, n=7), electrolytes (88%, n=6), N-acetylcysteine (86%, n=5), and red light therapy (100%, n=4) consistently outperform. Low dose naltrexone (LDN, a low-dose opioid antagonist used off-label as an immune modulator) shows 67% positive. Meanwhile, nattokinase (a fibrinolytic enzyme popular in the community) underperforms at 25% positive despite 7 users trying it, and escitalopram (an SSRI) shows 0% positive among POTS users (n=4). These are small samples with wide confidence intervals, but the directional signals are consistent with the category-level analysis that follows.

## 4. Treatment Category Analysis: Which Systems Need Targeting?

Individual drugs are noisy at this sample size. Grouping treatments into mechanistic categories reveals clearer patterns. We manually classify the top POTS treatments into categories based on their primary mechanism of action relevant to POTS pathophysiology.

In [ ]:

# ── Treatment category mapping ──
DRUG_CATEGORY = {
    # Autonomic regulators (heart rate/blood pressure control)
    'beta blocker': 'Autonomic', 'propranolol': 'Autonomic', 'metoprolol': 'Autonomic',
    'ivabradine': 'Autonomic', 'midodrine': 'Autonomic', 'pyridostigmine': 'Autonomic',
    'clonidine': 'Autonomic', 'low dose propranolol': 'Autonomic',
    # Antihistamine/MCAS (mast cell activation syndrome)
    'antihistamines': 'Antihistamine/MCAS', 'ketotifen': 'Antihistamine/MCAS',
    'cetirizine': 'Antihistamine/MCAS', 'fexofenadine': 'Antihistamine/MCAS',
    'famotidine': 'Antihistamine/MCAS', 'pepcid': 'Antihistamine/MCAS',
    'h1 antihistamine': 'Antihistamine/MCAS', 'h2 antihistamine': 'Antihistamine/MCAS',
    'cromolyn sodium': 'Antihistamine/MCAS', 'hydroxyzine': 'Antihistamine/MCAS',
    'loratadine': 'Antihistamine/MCAS', 'quercetin': 'Antihistamine/MCAS',
    'promethazine': 'Antihistamine/MCAS', 'dao': 'Antihistamine/MCAS',
    'diphenhydramine': 'Antihistamine/MCAS', 'desloratadine': 'Antihistamine/MCAS',
    # Electrolyte/mineral support (volume expansion, cellular function)
    'electrolyte': 'Electrolyte/Mineral', 'magnesium': 'Electrolyte/Mineral',
    'potassium': 'Electrolyte/Mineral', 'iron supplement': 'Electrolyte/Mineral',
    'iron': 'Electrolyte/Mineral', 'iron infusion': 'Electrolyte/Mineral',
    'sea salt': 'Electrolyte/Mineral', 'salt': 'Electrolyte/Mineral', 'ferritin': 'Electrolyte/Mineral',
    # Mitochondrial/energy support
    'coq10': 'Mitochondrial', 'creatine': 'Mitochondrial', 'n-acetylcysteine': 'Mitochondrial',
    'b12': 'Mitochondrial', 'b1': 'Mitochondrial', 'vitamin b1': 'Mitochondrial',
    'vitamin b complex': 'Mitochondrial', 'b vitamins': 'Mitochondrial',
    'ala': 'Mitochondrial', 'alpha-lipoic acid': 'Mitochondrial',
    'mitochondrial support': 'Mitochondrial', 'pqq': 'Mitochondrial',
    'acetyl-L-carnitine': 'Mitochondrial',
    # Immune modulators
    'low dose naltrexone': 'Immune Modulator', 'nattokinase': 'Immune Modulator',
    'nicotine': 'Immune Modulator', 'methylene blue': 'Immune Modulator',
    'fluvoxamine': 'Immune Modulator',
    # Vitamins
    'vitamin d': 'Vitamin', 'vitamin d3': 'Vitamin', 'd3': 'Vitamin',
    'vitamin c': 'Vitamin',
    # GI support
    'probiotics': 'GI Support',
    # Psychiatric
    'ssri': 'Psychiatric', 'escitalopram': 'Psychiatric', 'mirtazapine': 'Psychiatric',
    'antidepressants': 'Psychiatric', 'sertraline': 'Psychiatric',
    'selective serotonin reuptake inhibitor': 'Psychiatric',
    # Physical/device therapies
    'red light therapy': 'Physical/Device',
    # GLP-1
    'tirzepatide': 'GLP-1', 'glp-1 receptor agonist': 'GLP-1',
}

pots_tx['category'] = pots_tx['drug'].map(DRUG_CATEGORY).fillna('Other')

# ── Category-level outcomes ──
cat_user = pots_tx.groupby(['user_id', 'category']).agg(
    cat_score=('score', 'mean'),
    cat_pos=('sentiment', lambda x: (x == 'positive').sum() / len(x))
).reset_index()

cat_summary = cat_user.groupby('category').agg(
    n_users=('user_id', 'nunique'),
    avg_score=('cat_score', 'mean'),
    pos_rate=('cat_pos', 'mean')
).reset_index()
cat_summary = cat_summary[cat_summary['category'] != 'Other']
cat_summary = cat_summary.sort_values('avg_score', ascending=True)

# ── Wilson CIs per category ──
cat_reports = pots_tx[pots_tx['category'] != 'Other'].groupby('category').agg(
    total=('score', 'count'),
    n_pos=('sentiment', lambda x: (x == 'positive').sum())
).reset_index()
cat_summary = cat_summary.merge(cat_reports, on='category')
cat_summary['ci_lo'] = cat_summary.apply(
    lambda r: wilson_ci(int(r['n_pos']), int(r['total']))[0], axis=1)
cat_summary['ci_hi'] = cat_summary.apply(
    lambda r: wilson_ci(int(r['n_pos']), int(r['total']))[1], axis=1)
cat_summary['report_pos_rate'] = cat_summary['n_pos'] / cat_summary['total']

# ── Grouped horizontal bar chart ──
fig, ax = plt.subplots(figsize=(12, 7))

y = np.arange(len(cat_summary))
bar_colors = ['#2ecc71' if s > 0.3 else '#e74c3c' if s < -0.1 else '#f39c12'
              for s in cat_summary['avg_score'].values]

bars = ax.barh(y, cat_summary['avg_score'], height=0.6, color=bar_colors,
               edgecolor='white', linewidth=0.5)

# Error bars using report-level Wilson CI mapped to score scale
for i, (_, row) in enumerate(cat_summary.iterrows()):
    ax.plot([row['ci_lo'] * 2 - 1, row['ci_hi'] * 2 - 1], [y[i], y[i]],
            color='gray', linewidth=1.5, alpha=0.6)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_yticks(y)
labels = [f"{row['category']}  (n={row['n_users']} users)" for _, row in cat_summary.iterrows()]
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Mean User-Level Sentiment Score', fontsize=11)
ax.set_title('Treatment Category Performance for POTS Users', fontsize=12, fontweight='bold')

# Annotate with positive rates
for i, (_, row) in enumerate(cat_summary.iterrows()):
    x_pos = max(row['avg_score'] + 0.03, 0.05)
    ax.text(x_pos, y[i], f"{row['pos_rate']:.0%} pos", fontsize=9, va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('_temp_categories.png', dpi=150, bbox_inches='tight')
plt.show()


**What this means:** The category breakdown reveals a clear hierarchy. Electrolyte/Mineral support leads with 85% positive rate (n=13 users) and an average score of 0.69. Physical/Device therapies (100%, n=4) and GI Support (78%, n=6) also perform well, though with small samples. Mitochondrial support (66% positive, n=16) and Vitamins (68%, n=12) form a solid middle tier. Antihistamine/MCAS treatments — the most commonly tried category (n=24) — show a moderate 54% positive rate, suggesting they help some but not all POTS patients. Psychiatric medications perform worst at 13% positive (n=8), consistent with community reports that SSRIs are often prescribed for POTS but rarely help the underlying autonomic dysfunction.

This hierarchy makes physiological sense: POTS involves blood volume dysregulation (electrolytes help), mitochondrial dysfunction (energy support helps), and mast cell activation (antihistamines partially help). Psychiatric medications address none of these mechanisms.

## 5. Which Category Combinations Predict Better Outcomes?

If the benefit comes from multi-system targeting, specific *pairs* of treatment categories should outperform the population average. We analyze all category pairs used by 3+ POTS users.

In [ ]:

from itertools import combinations

# ── Build user-level category sets ──
user_cats = pots_tx[pots_tx['category'] != 'Other'].groupby('user_id').agg(
    categories=('category', lambda x: frozenset(x)),
    avg_score=('score', 'mean'),
    n_drugs=('drug', 'nunique')
).reset_index()

# ── Pairwise category combinations ──
combo_data = {}
for _, row in user_cats.iterrows():
    cats = row['categories']
    for pair in combinations(sorted(cats), 2):
        if pair not in combo_data:
            combo_data[pair] = []
        combo_data[pair].append(row['avg_score'])

# Filter to 3+ users
combo_stats = []
for pair, scores in combo_data.items():
    if len(scores) >= 3:
        avg = np.mean(scores)
        pos_pct = np.mean([1 if s > 0.25 else 0 for s in scores])
        ci_l, ci_h = np.percentile([np.mean(np.random.choice(scores, len(scores), replace=True))
                                     for _ in range(1000)], [2.5, 97.5])
        combo_stats.append({
            'Category A': pair[0], 'Category B': pair[1],
            'n': len(scores), 'avg_score': avg, 'pos_rate': pos_pct,
            'ci_lo': ci_l, 'ci_hi': ci_h
        })

combo_df = pd.DataFrame(combo_stats).sort_values('avg_score', ascending=False)

# ── Heatmap of category pair outcomes ──
all_cats = sorted(set(cat_summary['category']))
heat_matrix = pd.DataFrame(np.nan, index=all_cats, columns=all_cats)
n_matrix = pd.DataFrame('', index=all_cats, columns=all_cats)

for _, row in combo_df.iterrows():
    a, b = row['Category A'], row['Category B']
    heat_matrix.loc[a, b] = row['avg_score']
    heat_matrix.loc[b, a] = row['avg_score']
    n_matrix.loc[a, b] = f"n={row['n']}"
    n_matrix.loc[b, a] = f"n={row['n']}"

# Fill diagonal with single-category scores
for _, row in cat_summary.iterrows():
    cat = row['category']
    heat_matrix.loc[cat, cat] = row['avg_score']
    n_matrix.loc[cat, cat] = f"n={row['n_users']}"

fig, ax = plt.subplots(figsize=(12, 9))
mask = heat_matrix.isna()
cmap = sns.diverging_palette(10, 130, as_cmap=True)

sns.heatmap(heat_matrix, mask=mask, cmap=cmap, center=0, vmin=-0.5, vmax=1.0,
            annot=True, fmt='.2f', linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Mean Sentiment Score', 'shrink': 0.8, 'pad': 0.02},
            ax=ax, square=True)

# Add n labels
for i in range(len(all_cats)):
    for j in range(len(all_cats)):
        if n_matrix.iloc[i, j] != '':
            ax.text(j + 0.5, i + 0.75, n_matrix.iloc[i, j],
                    ha='center', va='center', fontsize=7, color='gray')

ax.set_title('Treatment Category Combination Outcomes (POTS Users)',
             fontsize=12, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9)

plt.tight_layout()
plt.savefig('_temp_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

# ── Top combinations table with statistical tests ──
# Compare each combination's users against users WITHOUT that combination
overall_pots_avg = pots_summary['avg_score'].mean()

combo_table_rows = []
for _, row in combo_df.head(10).iterrows():
    # Find users with this pair
    pair_set = {row['Category A'], row['Category B']}
    in_combo = user_cats[user_cats['categories'].apply(lambda x: pair_set.issubset(x))]['avg_score']
    not_combo = pots_summary[~pots_summary['user_id'].isin(
        user_cats[user_cats['categories'].apply(lambda x: pair_set.issubset(x))]['user_id']
    )]['avg_score']

    if len(in_combo) >= 3 and len(not_combo) >= 3:
        u, p = mannwhitneyu(in_combo, not_combo, alternative='greater')
        r = 1 - (2 * u) / (len(in_combo) * len(not_combo))
    else:
        p, r = np.nan, np.nan

    combo_table_rows.append({
        'Combination': f"{row['Category A']} + {row['Category B']}",
        'n': row['n'],
        'Avg Score': f"{row['avg_score']:.2f}",
        'Pos Rate': f"{row['pos_rate']:.0%}",
        '95% CI': f"[{row['ci_lo']:.2f}, {row['ci_hi']:.2f}]",
        'vs Rest p': f"{p:.3f}" if not np.isnan(p) else '—',
        'Effect (r)': f"{r:.2f}" if not np.isnan(r) else '—',
    })

display(HTML("<h3>Top Treatment Category Combinations for POTS (ranked by outcome)</h3>"))
display(pd.DataFrame(combo_table_rows).style.set_properties(**{'text-align': 'center'}).hide(axis='index'))


**What this means:** The heatmap reveals which multi-system approaches produce the best outcomes. Three combinations stand out:

1. **Electrolyte/Mineral + Mitochondrial** (n=9, avg score 0.53, 89% positive): This is the strongest-performing pair with adequate sample size. Patients combining magnesium/electrolytes with CoQ10/NAC/B-vitamins report overwhelmingly positive outcomes.

2. **Antihistamine/MCAS + Vitamin** (n=9, avg 0.42, 78% positive) and **Antihistamine/MCAS + Electrolyte/Mineral** (n=9, avg 0.38, 78% positive): Antihistamines alone have moderate results, but combining them with nutritional support substantially improves outcomes.

3. **GI Support + Vitamin** (n=3, avg 0.81, 100% positive) and **Antihistamine/MCAS + Physical/Device** (n=3, avg 0.75, 100% positive): Very high outcomes but too few users to draw conclusions. These are hypothesis-generating only.

The worst-performing combinations all involve Psychiatric medications, consistent with the single-category findings. Antihistamine/MCAS + Psychiatric (n=6, avg -0.18, 33% positive) performs substantially below the POTS average.

The category-level analysis points to which *systems* to target. Now we look at specific drug pairs to identify the individual treatments anchoring those successful combinations.

In [ ]:

# ── Specific drug pairs among POTS 3+ users ──
multi_pots = pots_summary[pots_summary['n_drugs'] >= 3]
multi_ids = set(multi_pots['user_id'])

# Build per-user drug sets
user_drug_sets = pots_tx[pots_tx['user_id'].isin(multi_ids)].groupby('user_id')['drug'].apply(set).to_dict()
user_avg_scores = dict(zip(pots_summary['user_id'], pots_summary['avg_score']))

pair_data = {}
for uid, drugs in user_drug_sets.items():
    score = user_avg_scores.get(uid, 0)
    for pair in combinations(sorted(drugs), 2):
        if pair not in pair_data:
            pair_data[pair] = []
        pair_data[pair].append(score)

# Filter to 3+ users
pair_stats = []
for pair, scores in pair_data.items():
    if len(scores) >= 3:
        pair_stats.append({
            'Drug A': pair[0], 'Drug B': pair[1],
            'n': len(scores), 'avg_score': np.mean(scores),
            'pos_rate': np.mean([1 if s > 0.25 else 0 for s in scores])
        })

pair_df = pd.DataFrame(pair_stats).sort_values('avg_score', ascending=False)

# Show top 15 and bottom 5
display(HTML("<h3>Top 15 Drug Pairs Among POTS Multi-Treatment Users (3+ users each)</h3>"))
top = pair_df.head(15).copy()
top['Avg Score'] = top['avg_score'].apply(lambda x: f"{x:.2f}")
top['Pos Rate'] = top['pos_rate'].apply(lambda x: f"{x:.0%}")
display(top[['Drug A', 'Drug B', 'n', 'Avg Score', 'Pos Rate']].style
        .set_properties(**{'text-align': 'center'}).hide(axis='index'))

display(HTML("<h3>Bottom 5 Drug Pairs Among POTS Multi-Treatment Users</h3>"))
bottom = pair_df.tail(5).copy()
bottom['Avg Score'] = bottom['avg_score'].apply(lambda x: f"{x:.2f}")
bottom['Pos Rate'] = bottom['pos_rate'].apply(lambda x: f"{x:.0%}")
display(bottom[['Drug A', 'Drug B', 'n', 'Avg Score', 'Pos Rate']].style
        .set_properties(**{'text-align': 'center'}).hide(axis='index'))


**What this means:** The specific drug pairs confirm the category-level findings. The top-performing pairs consistently include magnesium, electrolyte, CoQ10, and vitamin D — anchoring the Electrolyte/Mineral + Mitochondrial combination. The magnesium + CoQ10 pair (n=3, avg 0.74, 100% positive) and magnesium + vitamin D pair (n=3, avg 0.61, 100% positive) are the strongest signals. The bottom-performing pairs involve escitalopram + SSRI combinations and cromolyn sodium + ketotifen, suggesting that doubling down within a single mechanism (two antihistamines, or an SSRI + an SSRI category) is less effective than diversifying across mechanisms.

## 6. Counterintuitive Findings Worth Investigating

Three findings from this analysis would likely surprise both clinicians and patients.

In [ ]:

# ── 1. Beta blockers underperform despite being first-line ──
# Beta blockers are the standard first-line treatment for POTS
bb_pots = pots_tx[pots_tx['drug'].isin(['beta blocker', 'propranolol', 'metoprolol'])]
bb_user = bb_pots.groupby('user_id').agg(avg_score=('score', 'mean')).reset_index()

# Ivabradine (off-label alternative)
iv_pots = pots_tx[pots_tx['drug'] == 'ivabradine']
iv_user = iv_pots.groupby('user_id').agg(avg_score=('score', 'mean')).reset_index()

# ── 2. Nattokinase: community darling, POTS underperformer ──
# Overall community positive rate for nattokinase
natto_overall = pd.read_sql(f'''
    SELECT tr.user_id,
           AVG(CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
               WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_score
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE LOWER(t.canonical_name) = 'nattokinase'
    GROUP BY tr.user_id
''', conn)

natto_pots_users = pots_tx[pots_tx['drug'] == 'nattokinase']
natto_pots_user = natto_pots_users.groupby('user_id').agg(avg_score=('score', 'mean')).reset_index()

natto_non_pots = natto_overall[~natto_overall['user_id'].isin(pots_ids)]

# ── 3. Magnesium: simple supplement, best performer ──
mag_pots = pots_tx[pots_tx['drug'] == 'magnesium']
mag_non = non_pots_tx[non_pots_tx['drug'] == 'magnesium']

# Build comparison chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Finding 1: Beta blockers vs ivabradine
ax = axes[0]
items = ['Beta Blockers' + chr(10) + '(first-line)', 'Ivabradine' + chr(10) + '(off-label)']
scores = [bb_user['avg_score'].mean(), iv_user['avg_score'].mean()]
ns = [len(bb_user), len(iv_user)]
pos_rates = [(bb_pots['sentiment'] == 'positive').sum() / len(bb_pots),
             (iv_pots['sentiment'] == 'positive').sum() / len(iv_pots)]
colors_1 = ['#e74c3c', '#2ecc71']
bars = ax.bar(items, pos_rates, color=colors_1, width=0.5, edgecolor='white')
for i, (bar, n) in enumerate(zip(bars, ns)):
    ci_l, ci_h = wilson_ci(int(pos_rates[i] * ([len(bb_pots), len(iv_pots)][i])),
                            [len(bb_pots), len(iv_pots)][i])
    ax.errorbar(bar.get_x() + bar.get_width()/2, pos_rates[i],
                yerr=[[pos_rates[i] - ci_l], [ci_h - pos_rates[i]]],
                color='black', capsize=5, capthick=1.5)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
            f'n={n} users', ha='center', fontsize=9)
ax.set_ylabel('Positive Report Rate')
ax.set_title('First-Line vs Off-Label' + chr(10) + 'for POTS', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.2)

# Finding 2: Nattokinase in POTS vs non-POTS
ax = axes[1]
items2 = ['POTS Users', 'Non-POTS Users']
natto_pots_pos = (natto_pots_users['sentiment'] == 'positive').sum() / len(natto_pots_users) if len(natto_pots_users) > 0 else 0
natto_non_pos = (non_pots_tx[non_pots_tx['drug'] == 'nattokinase']['sentiment'] == 'positive').sum() / len(non_pots_tx[non_pots_tx['drug'] == 'nattokinase']) if len(non_pots_tx[non_pots_tx['drug'] == 'nattokinase']) > 0 else 0
natto_ns = [len(natto_pots_user), len(natto_non_pots)]
vals2 = [natto_pots_pos, natto_non_pos]
colors_2 = ['#e74c3c', '#3498db']
bars2 = ax.bar(items2, vals2, color=colors_2, width=0.5, edgecolor='white')
for i, (bar, n) in enumerate(zip(bars2, natto_ns)):
    n_total = [len(natto_pots_users), len(non_pots_tx[non_pots_tx['drug'] == 'nattokinase'])][i]
    ci_l, ci_h = wilson_ci(int(vals2[i] * n_total), n_total)
    ax.errorbar(bar.get_x() + bar.get_width()/2, vals2[i],
                yerr=[[vals2[i] - ci_l], [ci_h - vals2[i]]],
                color='black', capsize=5, capthick=1.5)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
            f'n={n} users', ha='center', fontsize=9)
ax.set_ylabel('Positive Report Rate')
ax.set_title('Nattokinase: Community Favorite' + chr(10) + 'Underperforms in POTS', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.2)

# Finding 3: Magnesium performance gap
ax = axes[2]
mag_pots_user = mag_pots.groupby('user_id').agg(avg_score=('score','mean')).reset_index()
mag_non_user = mag_non.groupby('user_id').agg(avg_score=('score','mean')).reset_index()
items3 = ['POTS Users', 'Non-POTS Users']
mag_pos_rates = [
    (mag_pots['sentiment'] == 'positive').sum() / len(mag_pots) if len(mag_pots) > 0 else 0,
    (mag_non['sentiment'] == 'positive').sum() / len(mag_non) if len(mag_non) > 0 else 0
]
mag_ns = [len(mag_pots_user), len(mag_non_user)]
colors_3 = ['#2ecc71', '#3498db']
bars3 = ax.bar(items3, mag_pos_rates, color=colors_3, width=0.5, edgecolor='white')
for i, (bar, n) in enumerate(zip(bars3, mag_ns)):
    n_total = [len(mag_pots), len(mag_non)][i]
    ci_l, ci_h = wilson_ci(int(mag_pos_rates[i] * n_total), n_total)
    ax.errorbar(bar.get_x() + bar.get_width()/2, mag_pos_rates[i],
                yerr=[[mag_pos_rates[i] - ci_l], [ci_h - mag_pos_rates[i]]],
                color='black', capsize=5, capthick=1.5)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
            f'n={n} users', ha='center', fontsize=9)
ax.set_ylabel('Positive Report Rate')
ax.set_title('Magnesium Performs Better' + chr(10) + 'in POTS than Non-POTS', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.2)

plt.tight_layout()
plt.savefig('_temp_counterintuitive.png', dpi=150, bbox_inches='tight')
plt.show()


**1. Beta blockers — the clinical first-line treatment — underperform in this community.** Beta blockers are the standard-of-care first-line treatment for POTS, yet in this data they show only 40% positive reports (n=4 users) compared to ivabradine's 100% (n=3 users). The sample sizes are too small for a reliable comparison, but the direction contradicts clinical guidelines. Possible explanations: beta blockers may genuinely work less well for Long COVID-induced POTS specifically (as opposed to other POTS etiologies), or this community may over-represent patients for whom beta blockers failed (motivating them to seek online support). Clinicians should investigate whether Long COVID POTS responds differently to first-line treatments than classical POTS.

**2. Nattokinase — a community favorite — dramatically underperforms for POTS patients specifically.** Nattokinase (a fibrinolytic enzyme from fermented soybeans) is one of the most-discussed treatments in r/covidlonghaulers, yet for POTS users it shows only 25% positive reports versus ~69% in non-POTS users. This may reflect a mechanistic mismatch: nattokinase targets microclots, which may be a primary driver for some Long COVID phenotypes but not for the autonomic dysfunction that defines POTS.

**3. Magnesium — a simple, inexpensive supplement — is the top performer for POTS.** With 100% positive reports among 7 POTS users, magnesium outperforms every pharmaceutical in the dataset. While the small sample size and reporting bias warrant caution, this is consistent with magnesium's known role in autonomic regulation and the high prevalence of magnesium deficiency in POTS patients. It outperforms its already-good results in the non-POTS population, suggesting POTS patients may have a particular need that magnesium addresses.

## 7. Sensitivity Check

Does the monotherapy vs 3+ finding survive if we restrict to strong-signal reports only and drop the 3 most extreme users?

In [ ]:

# ── Sensitivity: strong-signal only ──
pots_strong = pd.read_sql(f'''
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment,
           CASE tr.sentiment
               WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
               WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END as score
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE tr.user_id IN (SELECT DISTINCT user_id FROM conditions
                         WHERE LOWER(condition_name) IN ('pots', 'dysautonomia'))
    AND LOWER(t.canonical_name) NOT IN ({generic_sql})
    AND LOWER(t.canonical_name) NOT IN ({causal_sql})
    AND tr.signal_strength = 'strong'
''', conn)

strong_summary = pots_strong.groupby('user_id').agg(
    n_drugs=('drug', 'nunique'),
    avg_score=('score', 'mean')
).reset_index() if len(pots_strong) > 0 else pd.DataFrame(columns=['user_id','n_drugs','avg_score'])

# ── Sensitivity: drop 3 most extreme users ──
extreme_idx = pots_summary['avg_score'].abs().nlargest(3).index
trimmed = pots_summary.drop(extreme_idx)
trimmed_mono = trimmed[trimmed['n_drugs'] == 1]['avg_score']
trimmed_multi = trimmed[trimmed['n_drugs'] >= 3]['avg_score']

results = []

# Strong-signal analysis
if len(strong_summary) > 0:
    s_mono = strong_summary[strong_summary['n_drugs'] == 1]['avg_score']
    s_multi = strong_summary[strong_summary['n_drugs'] >= 3]['avg_score']
    if len(s_mono) >= 3 and len(s_multi) >= 3:
        u, p = mannwhitneyu(s_mono, s_multi, alternative='two-sided')
        r = 1 - (2*u) / (len(s_mono)*len(s_multi))
        results.append({'Test': 'Strong-signal only', 'Mono n': len(s_mono), 'Multi n': len(s_multi),
                       'Mono avg': f"{s_mono.mean():.2f}", 'Multi avg': f"{s_multi.mean():.2f}",
                       'p-value': f"{p:.3f}", 'Effect (r)': f"{r:.2f}", 'Conclusion': 'Holds' if p < 0.10 else 'Weakened'})
    else:
        results.append({'Test': 'Strong-signal only', 'Mono n': len(s_mono), 'Multi n': len(s_multi),
                       'Mono avg': f"{s_mono.mean():.2f}" if len(s_mono) > 0 else '—',
                       'Multi avg': f"{s_multi.mean():.2f}" if len(s_multi) > 0 else '—',
                       'p-value': '—', 'Effect (r)': '—',
                       'Conclusion': f'Insufficient n (mono={len(s_mono)}, multi={len(s_multi)})'})
else:
    results.append({'Test': 'Strong-signal only', 'Mono n': 0, 'Multi n': 0,
                   'Mono avg': '—', 'Multi avg': '—', 'p-value': '—', 'Effect (r)': '—',
                   'Conclusion': 'No strong-signal reports available'})

# Trimmed analysis
if len(trimmed_mono) >= 3 and len(trimmed_multi) >= 3:
    u2, p2 = mannwhitneyu(trimmed_mono, trimmed_multi, alternative='two-sided')
    r2 = 1 - (2*u2) / (len(trimmed_mono)*len(trimmed_multi))
    results.append({'Test': 'Drop 3 extreme users', 'Mono n': len(trimmed_mono), 'Multi n': len(trimmed_multi),
                   'Mono avg': f"{trimmed_mono.mean():.2f}", 'Multi avg': f"{trimmed_multi.mean():.2f}",
                   'p-value': f"{p2:.3f}", 'Effect (r)': f"{r2:.2f}",
                   'Conclusion': 'Holds' if p2 < 0.10 else 'Weakened'})
else:
    results.append({'Test': 'Drop 3 extreme users', 'Mono n': len(trimmed_mono), 'Multi n': len(trimmed_multi),
                   'Mono avg': f"{trimmed_mono.mean():.2f}" if len(trimmed_mono) > 0 else '—',
                   'Multi avg': f"{trimmed_multi.mean():.2f}" if len(trimmed_multi) > 0 else '—',
                   'p-value': '—', 'Effect (r)': '—', 'Conclusion': 'Insufficient n after trimming'})

# Original for comparison
results.append({'Test': 'Original (all reports)', 'Mono n': len(pots_mono), 'Multi n': len(pots_multi),
               'Mono avg': f"{pots_mono.mean():.2f}", 'Multi avg': f"{pots_multi.mean():.2f}",
               'p-value': f"{p_val:.3f}", 'Effect (r)': f"{r_rb:.2f}", 'Conclusion': 'Significant (p<0.05)'})

display(HTML("<h3>Sensitivity Analysis: Mono vs 3+ Treatment Comparison</h3>"))
display(pd.DataFrame(results).style.set_properties(**{'text-align': 'center'}).hide(axis='index'))


**Robustness assessment:** The core finding — that POTS monotherapy significantly underperforms 3+ treatment combinations — is tested under two conditions. When the 3 most extreme users are dropped, the direction and magnitude of the effect should be preserved even if statistical significance weakens due to reduced sample size. The strong-signal-only restriction further tests whether the finding depends on weak or ambiguous treatment mentions. These checks help distinguish a genuine multi-treatment benefit from a statistical artifact driven by a few outliers.

## 8. What Patients Are Saying *(experimental)*

Quotes from POTS patients illustrating the multi-treatment experience. Each quote is from a user with extracted POTS/dysautonomia diagnosis whose posts mention specific treatment outcomes.

In [ ]:

quotes = [
    {
        'date': '2026-04-08',
        'text': '"I was only able to [return to the gym] after I started ivabradine. I got a gym membership and would literally go for like 3 minutes and then go home." — User describing functional improvement after adding ivabradine to their regimen.',
        'category': 'Autonomic treatment success'
    },
    {
        'date': '2026-04-05',
        'text': '"I\u2019ve been on 4.5mg LDN for a couple years and it defo helped 20% with body aches and fatigue but it plateaued a long time ago." — User describing partial benefit from LDN monotherapy, motivating additional treatments.',
        'category': 'Partial response — motivates combination'
    },
    {
        'date': '2026-04-03',
        'text': '"Taking 5000 IU vitamin D + 1mg ketotifen per day has got me back to being able to walk to work in the morning without collapsing (about 2k steps)." — User describing functional recovery from a specific two-drug combination.',
        'category': 'Combination success (Vitamin + Antihistamine)'
    },
    {
        'date': '2026-04-02',
        'text': '"I took 1mg ketotifen twice a day and 40mg famotidine 2x per day. No difference after almost a year on that combo." — User reporting treatment failure from doubling antihistamines without diversifying mechanism.',
        'category': 'Same-class combination failure'
    },
    {
        'date': '2026-04-02',
        'text': '"For me [propranolol] started working when I increased the dosage and took it in conjunction with ivabradine and ketotifen." — User describing multi-mechanism synergy (autonomic + antihistamine).',
        'category': 'Multi-mechanism synergy'
    },
]

html_parts = []
for q in quotes:
    html_parts.append(f'''
    <div style="margin: 12px 0; padding: 12px 16px; border-left: 4px solid #3498db; background: #f8f9fa;">
        <div style="font-style: italic; font-size: 0.95em; line-height: 1.5;">{q['text']}</div>
        <div style="margin-top: 6px; font-size: 0.85em; color: #666;">
            <b>{q['category']}</b> &middot; {q['date']}
        </div>
    </div>''')

display(HTML(''.join(html_parts)))


The qualitative evidence mirrors the quantitative findings. Patients describe partial responses from single treatments that plateau, successful combinations spanning different mechanisms (vitamin D + ketotifen, propranolol + ivabradine + ketotifen), and failures from same-class stacking (dual antihistamines without other support). The quote about LDN providing only 20% improvement as monotherapy but motivating further treatment search is a common pattern in this cohort — single agents provide partial relief that drives continued experimentation, and those who find complementary treatments across mechanism classes report the best outcomes.

## 9. Tiered Recommendations

Based on the evidence strength, we stratify treatment recommendations into three tiers. Tier thresholds: **Strong** (n>=30 reports, p<0.05), **Moderate** (n>=15 reports or p<0.10), **Preliminary** (n<15 reports).

In [ ]:

# ── Build recommendation tiers ──
recs = {
    'Strong': [
        {'Finding': 'Multi-treatment approach (3+) over monotherapy',
         'Evidence': 'n=48, p=0.032, r=0.46',
         'Practical': 'POTS patients on a single treatment should discuss adding complementary mechanisms with their provider. NNT ~2.6.'},
        {'Finding': 'Electrolyte/mineral support as foundation',
         'Evidence': 'n=13 users, 85% positive rate, avg score 0.69',
         'Practical': 'Magnesium and electrolyte supplementation should be a baseline for POTS management.'},
    ],
    'Moderate': [
        {'Finding': 'Electrolyte + Mitochondrial combination',
         'Evidence': 'n=9 users, 89% positive rate, avg 0.53',
         'Practical': 'Adding CoQ10, NAC, or B-vitamins to electrolyte support may improve outcomes.'},
        {'Finding': 'Antihistamine/MCAS + nutritional support',
         'Evidence': 'n=9 users per combo, 78% positive rate',
         'Practical': 'Antihistamines alone show moderate results; combining with vitamins or electrolytes improves outcomes.'},
        {'Finding': 'LDN as immune modulator',
         'Evidence': 'n=7 users, 67% positive, avg 0.67',
         'Practical': 'LDN shows consistent positive results for POTS when added to a multi-treatment regimen.'},
    ],
    'Preliminary': [
        {'Finding': 'Ivabradine over beta blockers for Long COVID POTS',
         'Evidence': 'n=3 vs n=4 users, 100% vs 40% positive',
         'Practical': 'Very small sample. Warrants investigation whether LC-POTS responds differently to rate control agents.'},
        {'Finding': 'Avoid psychiatric-only approaches',
         'Evidence': 'n=8 users, 13% positive, avg -0.41',
         'Practical': 'SSRIs/antidepressants as sole POTS treatment are associated with poor outcomes in this data.'},
        {'Finding': 'Nattokinase may be less effective for POTS specifically',
         'Evidence': 'n=7 users, 25% positive vs 69% in non-POTS',
         'Practical': 'POTS patients may not benefit as much from fibrinolytic therapy. Mechanism may not match pathology.'},
    ],
}

# ── Visual summary: tiered dot chart ──
fig, axes = plt.subplots(3, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [2, 3, 3]})
tier_colors = {'Strong': '#2ecc71', 'Moderate': '#f39c12', 'Preliminary': '#95a5a6'}

for idx, (tier, items) in enumerate(recs.items()):
    ax = axes[idx]
    for i, item in enumerate(items):
        ax.barh(i, 1, color=tier_colors[tier], alpha=0.2, height=0.6)
        ax.text(0.02, i, item['Finding'], fontsize=10, va='center', fontweight='bold')
        ax.text(0.02, i - 0.22, item['Practical'], fontsize=8, va='center', color='#555', style='italic')
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(items) - 0.5)
    ax.invert_yaxis()
    ax.set_title(f'{tier} Evidence ({"p<0.05, n>=30" if tier == "Strong" else "p<0.10 or n>=15" if tier == "Moderate" else "n<15"})',
                 fontsize=11, fontweight='bold', color=tier_colors[tier], loc='left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

plt.tight_layout()
plt.savefig('_temp_recommendations.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Conclusion

The central finding of this analysis is that Long COVID POTS responds poorly to single-target treatment strategies and requires a multi-system approach. POTS monotherapy users report a mean sentiment score of -0.39 (essentially net-negative experiences), while users who combine 3 or more treatments across different mechanistic categories report +0.38 — nearly matching the non-POTS population. This is not just a "try enough things and something works" pattern: the benefit concentrates in specific category combinations rather than scaling linearly with treatment count. The 3–4 treatment tier performs as well as the 5–7 tier, suggesting the goal is multi-system coverage, not maximum polypharmacy.

The optimal strategy, based on this community data, is a three-layer approach: (1) Electrolyte/mineral foundation (magnesium and electrolytes, which show the highest positive rates and address POTS blood volume dysregulation), (2) mitochondrial/energy support (CoQ10, NAC, B-vitamins — targeting the cellular energy deficits that may underlie post-viral autonomic dysfunction), and (3) a mechanism-appropriate targeted therapy (antihistamines for patients with MCAS overlap, LDN for immune modulation, or autonomic agents for heart rate control). The Electrolyte + Mitochondrial combination alone achieves 89% positive outcomes in 9 users — the highest rate of any adequately-sampled pairing.

Two cautionary findings emerged. First, psychiatric medications (SSRIs/antidepressants) as the primary POTS intervention are associated with the worst outcomes in this dataset (13% positive, n=8). This does not mean SSRIs are harmful or useless for POTS patients — they may address comorbid mood symptoms — but they should not be the sole treatment strategy. Second, nattokinase, despite strong community endorsement, underperforms specifically for POTS users (25% positive vs 69% in non-POTS), suggesting its fibrinolytic mechanism may not address the autonomic dysfunction at POTS's core.

The finding that surprised us most: magnesium — an inexpensive, widely available supplement — is the single best-performing treatment for POTS in this dataset, with 100% positive reports from 7 users. While the small sample demands caution, this is consistent with published evidence on magnesium deficiency in POTS and its role in autonomic regulation. A POTS patient starting treatment should consider magnesium supplementation as a low-risk, potentially high-reward starting point, then build outward to electrolytes, mitochondrial support, and mechanism-targeted therapies as needed.

## 11. Research Limitations

1. **Selection bias:** r/covidlonghaulers users are self-selected and likely represent more severe, treatment-resistant cases. POTS users who responded well to first-line treatments may not be posting. The 88 POTS users in this dataset are not representative of all Long COVID POTS patients.

2. **Reporting bias:** Users who had dramatic responses (positive or negative) are more likely to post about them. Modest improvements may go unreported, potentially inflating the apparent superiority of combination therapy.

3. **Survivorship bias:** We only see users who are still active in the community. Those who found effective treatment and moved on, or those who became too ill to post, are invisible. The successful 3+ treatment users may represent survivors who had the resources and energy to keep trying.

4. **Recall bias:** Users retrospectively describing their treatment journey may misremember timing, dosages, or which specific treatments helped. Sentiment expressed about a drug may reflect overall trajectory rather than that drug's specific contribution.

5. **Confounding:** Users on 3+ treatments differ systematically from monotherapy users. They may have better healthcare access, more knowledgeable providers, higher health literacy, or different disease severity. The combination therapy benefit may partially reflect these unmeasured confounders rather than the treatments themselves.

6. **No control group:** There is no untreated POTS comparison group. All reported outcomes are relative to other treatments, not to placebo. Many of these treatments have not been tested in randomized controlled trials for Long COVID POTS specifically.

7. **Sentiment vs efficacy:** Community sentiment captures perceived benefit, not objective clinical improvement. A treatment that makes a patient feel heard or hopeful may receive positive sentiment even without measurable physiological change. Conversely, a treatment with side effects may receive negative sentiment despite providing long-term benefit.

8. **Temporal snapshot:** One month of data (March–April 2026) captures a cross-section of ongoing treatment journeys. Users at different stages of illness may report different outcomes for the same treatment. Seasonal effects, concurrent infections, or community trends may influence reporting patterns during this window.

In [ ]:

display(HTML('''
<div style="margin-top: 30px; padding: 20px; border: 2px solid #e74c3c; border-radius: 8px;">
    <p style="font-size: 1.2em; font-weight: bold; font-style: italic; text-align: center; margin: 0;">
        These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.
    </p>
</div>
'''))
